# sindyc_06: units in reach R as a state

Follows "Analysis plan: units in reach R as a state" in `docs/2026-11-11_status-and-next.md`.
This notebook covers the first part only: data analysis, no fitting.

- **0.5 In reach vs firing.** R is defined as "an enemy within ρ = 5.75" (the game's attack
  test, centre to centre). How often is a unit in reach but *not* shooting (its own target is
  out of range)?
- **1 Is a state needed?** Residual = measured R − closure A·Φ(…). Does it depend on history
  (time since contact, intent, direction of travel), and does it persist within a battle?

Data: the exp06 validation set (1000 battles, not used to fit the closure). The closure is
evaluated on the *measured* distance and formation at each step, so these residuals are the
closure's own error, separate from the distance error in rollouts.

exp06 is exploratory (collected with `allow_dirty`): re-run this after re-collection.

In [ ]:
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sindy_wm.io import storage
from sindy_wm.models.intent_model import SHAPE, STATE, add_controls, add_rates, team_shares
from sindy_wm.paths import DATA_DIR, RAW_DIR
from sindy_wm.states.geometry import REACH, REF_UNITS

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
PROCESSED = DATA_DIR / "processed" / "exp06_intents"
RAW = RAW_DIR / "exp06_intents" / "val"
MOVE_COLOR = {"advance": "tab:green", "hold": "tab:blue", "withdraw": "tab:red"}
SIDES = ("blue", "red")

current = pickle.load(open(PROCESSED / "model_derived.pkl", "rb"))
val = add_rates(add_controls(pd.read_parquet(PROCESSED / "val.parquet")), cols=("distance",))
print(f"{val.index.get_level_values(0).nunique()} battles, {len(val)} steps, ρ = {REACH}")

## 0.5 In reach (ρ = 5.75) vs firing (`target_in_range`)

Per unit and step: *in reach* = some living enemy within ρ; *firing* = alive and its own target
within attack range (shooting, or waiting for its cooldown). Firing without being in reach
should not happen, since both use the same range test; it is printed as a sanity check.

In [ ]:
snap = storage.load_snapshots(RAW)
snap = snap[snap["alive"]]
key = ["episode_idx", "step"]


def wide(team, col):
    return snap[snap["team"] == team].pivot_table(index=key, columns="unit_id", values=col, aggfunc="first")


xy = {t: (wide(t, "x"), wide(t, "y")) for t in SIDES}
steps = xy["blue"][0].index.union(xy["red"][0].index)
P = {t: np.stack([m.reindex(steps).to_numpy(float) for m in xy[t]], -1) for t in SIDES}  # (N, units, 2)
gap = np.linalg.norm(P["blue"][:, :, None, :] - P["red"][:, None, :, :], axis=-1)
within = gap <= REACH  # NaN (dead / absent) compares False
in_reach = {"blue": within.any(axis=2), "red": within.any(axis=1)}
firing = {t: wide(t, "target_in_range").reindex(steps).fillna(False).astype(bool).to_numpy() for t in SIDES}

unit_rows = []
for t in SIDES:
    unit_rows.append(
        pd.DataFrame(
            {
                "side": t,
                "in_reach": in_reach[t].sum(1),
                "firing": firing[t].sum(1),
                "reach_not_firing": (in_reach[t] & ~firing[t]).sum(1),
                "firing_not_reach": (~in_reach[t] & firing[t]).sum(1),
            },
            index=steps,
        )
    )
units = pd.concat(unit_rows).set_index("side", append=True)

# sanity: the in-reach count matches the processed table
check = units.xs("blue", level="side")["in_reach"].reindex(val.index) - REF_UNITS * val["reach_blue"]
print(f"max |in-reach count − processed reach_blue|: {check.abs().max():.2g}")
print(f"unit-steps firing without being in reach: {int(units['firing_not_reach'].sum())}")

In [ ]:
intent = val[["movement", "target"]]
u = units.join(intent, on=key)
table = (
    u.groupby(["side", "movement", "target"])[["in_reach", "reach_not_firing"]]
    .sum()
    .assign(**{"share not firing": lambda d: d["reach_not_firing"] / d["in_reach"]})
)
table.round(3)

The same gap over the course of a battle. Time is measured from first contact (the first step
where any unit on either side is in reach).

In [ ]:
contact_t = val["t"].where(val["reach_blue"] + val["reach_red"] > 0).groupby(level=0).transform("min")
val["since_contact"] = val["t"] - contact_t
u = u.join(val["since_contact"], on=key)
u["tbin"] = (u["since_contact"] / 0.25).round() * 0.25

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, side in zip(axes, SIDES, strict=True):
    for (mv, tg), g in u.xs(side, level="side").groupby(["movement", "target"]):
        m = g[g["tbin"].between(-1, 8)].groupby("tbin")[["in_reach", "reach_not_firing"]].mean()
        ls = "-" if tg == "focus" else "--"
        ax.plot(m.index, m["reach_not_firing"], ls, color=MOVE_COLOR[mv], label=f"{mv}/{tg}")
    ax.set(
        title=f"{side}: units in reach but not firing", xlabel="seconds since first contact", ylabel="units"
    )
axes[0].legend(fontsize=8)
plt.tight_layout()

## 1 Closure residuals: is a state needed?

Residual = measured R − A·s(d, depth, width), in units (×10 on the common scale), on steps where
both sides are alive. Positive = more units in reach than the closure says.

In [ ]:
rows = val[(val["alive_blue"] > 0) & (val["alive_red"] > 0)].copy()
sb, sr = team_shares(rows[STATE].to_numpy(), rows[SHAPE].to_numpy(), current.P)
for side, s in (("blue", sb), ("red", sr)):
    rows[f"closure_{side}"] = REF_UNITS * rows[f"alive_{side}"] * s
    rows[f"R_{side}"] = REF_UNITS * rows[f"reach_{side}"]
    rows[f"res_{side}"] = rows[f"R_{side}"] - rows[f"closure_{side}"]

summary = pd.DataFrame(
    {
        side: {
            "mean residual": rows[f"res_{side}"].mean(),
            "mean |residual|": rows[f"res_{side}"].abs().mean(),
            "mean |residual|, R or closure > 0.5": rows.loc[
                (rows[f"R_{side}"] > 0.5) | (rows[f"closure_{side}"] > 0.5), f"res_{side}"
            ]
            .abs()
            .mean(),
        }
        for side in SIDES
    }
)
summary.round(3)

### 1a Against time since first contact, per movement

In [ ]:
rows["tbin"] = (rows["since_contact"] / 0.25).round() * 0.25


def binned(df, by, col, lo, hi, min_n=50):
    g = df[df[by].between(lo, hi)].groupby(by)[col]
    m = pd.DataFrame({"mean": g.mean(), "se": g.std() / np.sqrt(g.count()), "n": g.count()})
    return m[m["n"] >= min_n]


fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, side in zip(axes, SIDES, strict=True):
    for mv, g in rows.groupby("movement"):
        m = binned(g, "tbin", f"res_{side}", -2, 8)
        ax.plot(m.index, m["mean"], color=MOVE_COLOR[mv], label=mv)
        ax.fill_between(
            m.index, m["mean"] - 2 * m["se"], m["mean"] + 2 * m["se"], color=MOVE_COLOR[mv], alpha=0.2
        )
    ax.axhline(0, color="gray", lw=0.5)
    ax.axvline(0, color="gray", lw=0.5, ls=":")
    ax.set(title=f"R_{side} − closure", xlabel="seconds since first contact", ylabel="units (mean ± 2 se)")
axes[0].legend()
plt.tight_layout()

### 1b Hold vs advance at the same distance

Measured R and the closure against centroid distance, per movement. If the closure follows the
measured curve for each movement, distance and formation explain the difference; if the gap
differs by movement at the same distance, something else (history) does.

In [ ]:
rows["dbin"] = (rows["distance"] / 0.5).round() * 0.5
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for col, side in enumerate(SIDES):
    for mv, g in rows.groupby("movement"):
        meas = binned(g, "dbin", f"R_{side}", 2, 14)
        clos = binned(g, "dbin", f"closure_{side}", 2, 14)
        res = binned(g, "dbin", f"res_{side}", 2, 14)
        axes[0, col].plot(meas.index, meas["mean"], color=MOVE_COLOR[mv], label=f"{mv} measured")
        axes[0, col].plot(clos.index, clos["mean"], "--", color=MOVE_COLOR[mv], label=f"{mv} closure")
        axes[1, col].plot(res.index, res["mean"], color=MOVE_COLOR[mv], label=mv)
        axes[1, col].fill_between(
            res.index,
            res["mean"] - 2 * res["se"],
            res["mean"] + 2 * res["se"],
            color=MOVE_COLOR[mv],
            alpha=0.2,
        )
    axes[0, col].set(title=f"R_{side}", ylabel="units")
    axes[1, col].axhline(0, color="gray", lw=0.5)
    axes[1, col].set(title=f"R_{side} − closure", xlabel="centroid distance", ylabel="units (mean ± 2 se)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()

### 1c Same distance, different direction of travel

Hysteresis test: at the same distance, is R different when the teams are closing than when
they are moving apart? A function of the current state can't tell the two apart; a state with
memory can. Direction from the smoothed d′ (map units per second).

In [ ]:
rows["travel"] = pd.cut(
    rows["d_distance"],
    [-np.inf, -0.5, 0.5, np.inf],
    labels=["closing (d′ < −0.5)", "steady", "opening (d′ > 0.5)"],
)
TRAVEL_COLOR = dict(zip(rows["travel"].cat.categories, ["tab:purple", "tab:gray", "tab:orange"], strict=True))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, side in zip(axes, SIDES, strict=True):
    for tr, g in rows.groupby("travel", observed=True):
        m = binned(g, "dbin", f"res_{side}", 2, 14)
        ax.plot(m.index, m["mean"], color=TRAVEL_COLOR[tr], label=f"{tr} ({len(g)} steps)")
        ax.fill_between(
            m.index, m["mean"] - 2 * m["se"], m["mean"] + 2 * m["se"], color=TRAVEL_COLOR[tr], alpha=0.2
        )
    ax.axhline(0, color="gray", lw=0.5)
    ax.set(title=f"R_{side} − closure", xlabel="centroid distance", ylabel="units (mean ± 2 se)")
axes[0].legend(fontsize=8)
plt.tight_layout()

### 1d Does the residual persist within a battle?

Correlation between the residual now and k steps later, in the same battle (both sides alive at
both times). Only steps where R or the closure is above 0.5 units.

In [ ]:
lags = [1, 2, 4, 8, 16, 24, 32]
auto = {}
for side in SIDES:
    r = rows[f"res_{side}"].where((rows[f"R_{side}"] > 0.5) | (rows[f"closure_{side}"] > 0.5))
    full = r.reindex(val.index)  # keep step spacing: missing steps become NaN
    by_ep = full.groupby(level=0)
    auto[side] = {k * 0.125: full.corr(by_ep.shift(-k)) for k in lags}
auto = pd.DataFrame(auto)
auto.index.name = "lag (s)"
auto.round(2)

## Questions before step 2 (for you to answer, not answered here)

1. From 0.5: is the in-reach-but-not-firing gap big enough, and different enough between focus
   and nearest, that R (ρ = 5.75) and "units firing" should be treated as different things?
2. From 1a/1b: does the residual differ between hold and advance *at the same distance*? Which
   side's R does it affect?
3. From 1c: closing vs opening at the same distance. Is there hysteresis, and in which direction?
   What would cause it in the game?
4. From 1d: the residual persists for some time. Is that memory (history), or a slowly changing
   variable the closure is missing (formation)? Would R as a state fix either?
5. Taken together: is step 2 (splitting R′ into flows) justified?

## Test 1: is "units firing" F a better driver of health than "units in reach" R?

F = alive units whose own target is within attack range (`target_in_range`), counted per side
and step from the per-unit snapshots, on the common scale (1 = ten units). Counts are built by
`firing.py` into `firing_train.parquet` / `firing_val.parquet`.

**Game rule:** a marine does 6 damage every 0.61 s ≈ 9.8 per second. On the common scale
(health 1 = 450 hp, F 1 = ten units) that gives `H' = −10 · 9.8 / 450 · F_enemy ≈ −0.218 F_enemy`
for **both** sides. Prediction (written before running): F's coefficient comes closer to 0.218.

Fits on the training battles, R² on the validation battles, steps with both sides alive, the
same smoothed health rates as the model.

In [ ]:
from sindy_wm.models.intent_model import r2, stlsq  # noqa: E402

RULE = 10 * 6 / 0.61 / 450
print(f"game rule: {RULE:.3f} per unit of the common scale")


def with_firing(states, name):
    F = pd.read_parquet(PROCESSED / f"firing_{name}.parquet")
    out = states.join(F)
    out[["firing_blue", "firing_red"]] = out[["firing_blue", "firing_red"]].fillna(0)
    return out[(out["alive_blue"] > 0) & (out["alive_red"] > 0)]


tr = with_firing(
    add_rates(add_controls(pd.read_parquet(PROCESSED / "train.parquet")), cols=("health_blue", "health_red")),
    "train",
)
va = with_firing(
    add_rates(add_controls(pd.read_parquet(PROCESSED / "val.parquet")), cols=("health_blue", "health_red")),
    "val",
)
print(f"{len(tr)} training steps, {len(va)} validation steps")

### One term: `H_side' = −k · driver_enemy`

In [ ]:
one = []
for driver in ("reach", "firing"):
    for side, enemy in (("blue", "red"), ("red", "blue")):
        x, y = tr[f"{driver}_{enemy}"].to_numpy(), tr[f"d_health_{side}"].to_numpy()
        k = -(x @ y) / (x @ x)
        one.append(
            {
                "driver": "R (in reach)" if driver == "reach" else "F (firing)",
                "equation": f"H_{side}' = −k · {driver}_{enemy}",
                "k": k,
                "k / game rule": k / RULE,
                "R² (validation)": r2(va[f"d_health_{side}"], -k * va[f"{driver}_{enemy}"]),
            }
        )
pd.DataFrame(one).set_index(["driver", "equation"]).round(3)

### Same, but k fitted per intent

If the driver is the physics, k should be the same under every intent. If it absorbs a firing
share that depends on the intent, k will move.

In [ ]:
per = []
for (mv, tg), g in tr.groupby(["movement", "target"]):
    row = {"movement": mv, "target": tg}
    for driver in ("reach", "firing"):
        for side, enemy in (("blue", "red"), ("red", "blue")):
            x, y = g[f"{driver}_{enemy}"].to_numpy(), g[f"d_health_{side}"].to_numpy()
            row[f"{'R' if driver == 'reach' else 'F'}: H_{side}'"] = -(x @ y) / (x @ x)
    per.append(row)
per = pd.DataFrame(per).set_index(["movement", "target"])
per.loc[("spread", "max − min"), :] = per.max() - per.min()
per.round(3)

### With the model's control candidates

The same STLSQ fit as `model_derived` (threshold 0.05), library {driver, driver·hold,
driver·wd, driver·focus} of the enemy's driver. `model_derived` needed `+0.163 R_B·wd` in Red's
equation (withdrawing removes 88% of Blue's fire). Does F still need it?

In [ ]:
ctl = []
for driver in ("reach", "firing"):
    for side, enemy in (("blue", "red"), ("red", "blue")):

        def lib(df, driver=driver, enemy=enemy):
            x = df[f"{driver}_{enemy}"].to_numpy()
            return {
                "": x,
                "·hold": x * df["hold"].to_numpy(),
                "·wd": x * df["withdraw"].to_numpy(),
                "·focus": x * df["focus"].to_numpy(),
            }

        L, Lv = lib(tr), lib(va)
        c = stlsq(np.column_stack(list(L.values())), tr[f"d_health_{side}"].to_numpy(), 0.05)
        name = "R" if driver == "reach" else "F"
        terms = " ".join(f"{v:+.3f} {name}_{enemy[0]}{s}" for s, v in zip(L, c, strict=True) if v)
        ctl.append(
            {
                "driver": name,
                "equation": f"H_{side}' = {terms}",
                "terms": int((c != 0).sum()),
                "R² (validation)": r2(va[f"d_health_{side}"], np.column_stack(list(Lv.values())) @ c),
            }
        )
pd.DataFrame(ctl).set_index("driver")

### Fairness check: smoothing the drivers

The health rates are smoothed over 9 steps (Savitzky-Golay), but R and F above are raw. F
flickers more from step to step than R (a Blue unit's target is cleared by every move order, and
targets drop for a step when they die), which penalises F unfairly. Here both drivers get the same
9-step centred mean before fitting.

In [ ]:
for df in (tr, va):
    for c in ("firing_blue", "firing_red", "reach_blue", "reach_red"):
        df[c + "_sm"] = df.groupby(level=0)[c].transform(
            lambda x: x.rolling(9, center=True, min_periods=1).mean()
        )
flicker = pd.Series(
    {
        c: REF_UNITS * va.groupby(level=0)[c].diff().abs().mean()
        for c in ("reach_blue", "firing_blue", "reach_red", "firing_red")
    },
    name="mean |change per step| (units)",
)
display(flicker.round(3).to_frame())

sm = []
for driver in ("reach", "firing"):
    for suffix in ("", "_sm"):
        for side, enemy in (("blue", "red"), ("red", "blue")):
            col = f"{driver}_{enemy}{suffix}"
            x, y = tr[col].to_numpy(), tr[f"d_health_{side}"].to_numpy()
            k = -(x @ y) / (x @ x)
            sm.append(
                {
                    "driver": ("R" if driver == "reach" else "F") + (" smoothed" if suffix else " raw"),
                    "equation": f"H_{side}'",
                    "k": k,
                    "k / game rule": k / RULE,
                    "R² (validation)": r2(va[f"d_health_{side}"], -k * va[col]),
                }
            )
pd.DataFrame(sm).set_index(["driver", "equation"]).round(3)

Per intent, and with the control candidates, on the smoothed drivers.

In [ ]:
per_sm = []
for (mv, tg), g in tr.groupby(["movement", "target"]):
    row = {"movement": mv, "target": tg}
    for driver in ("reach", "firing"):
        for side, enemy in (("blue", "red"), ("red", "blue")):
            x, y = g[f"{driver}_{enemy}_sm"].to_numpy(), g[f"d_health_{side}"].to_numpy()
            row[f"{'R' if driver == 'reach' else 'F'}: H_{side}'"] = -(x @ y) / (x @ x)
    per_sm.append(row)
per_sm = pd.DataFrame(per_sm).set_index(["movement", "target"])
per_sm.loc[("spread", "max − min"), :] = per_sm.max() - per_sm.min()
display(per_sm.round(3))

ctl_sm = []
for driver in ("reach", "firing"):
    for side, enemy in (("blue", "red"), ("red", "blue")):

        def lib(df, driver=driver, enemy=enemy):
            x = df[f"{driver}_{enemy}_sm"].to_numpy()
            return {
                "": x,
                "·hold": x * df["hold"].to_numpy(),
                "·wd": x * df["withdraw"].to_numpy(),
                "·focus": x * df["focus"].to_numpy(),
            }

        L, Lv = lib(tr), lib(va)
        c = stlsq(np.column_stack(list(L.values())), tr[f"d_health_{side}"].to_numpy(), 0.05)
        name = "R" if driver == "reach" else "F"
        ctl_sm.append(
            {
                "driver": name + " smoothed",
                "equation": f"H_{side}' = "
                + " ".join(f"{v:+.3f} {name}_{enemy[0]}{s}" for s, v in zip(L, c, strict=True) if v),
                "R² (validation)": round(
                    r2(va[f"d_health_{side}"], np.column_stack(list(Lv.values())) @ c), 3
                ),
            }
        )
pd.DataFrame(ctl_sm).set_index("driver")

## Questions on test 1 (for you)

1. Your prediction was that F lands closer to 0.218. Raw vs smoothed: which comparison is the
   fair one, and why?
2. Compare the per-intent spread of k for R and F. Which one behaves like a constant of the game?
3. With F, the withdraw term leaves Red's equation, but a new one turns up in Blue's (Red does
   about 1.3× the game rule per firing unit when Blue withdraws). Since a unit can't do more than
   the rule, F must be *under*-counting Red's firing then. What could Red be doing at the
   snapshot moments while Blue runs away?
4. R has the higher R² for one side and F for the other. What would explain that?

## Test 2: does Red's F need a state, or is a share of R enough?

By the game rules, a Red unit picks up a target only within 5.375 and keeps it out to 5.75. So in
the 5.375–5.75 band, whether it fires should depend on where it has been: memory. Blue picks a
target fresh every step, so it should have none. Blue is the **negative control** throughout.

**2a, per unit (validation):** Red units in the band, split by whether they had an enemy within
5.375 at any point in the previous second (8 steps).

**2b, per team (train → validation):** predict smoothed F from the *current* team state with a
flexible model (gradient-boosted trees, so a share of any shape the state allows):

- `share`: F = φ · R (one number per side)
- `current state`: trees on R, A, distance, d′, depth, width of both sides and the intent
- `+ past R`: the same, plus R and distance 9 steps (1.1 s) earlier. This controls for "everything
  changes slowly".
- `+ past F`: the same, plus F 9 steps earlier

Memory = the R² gain from `+ past R` to `+ past F`. Nine steps keep the two 9-step smoothing windows
from overlapping. Caveat: past F can also stand in for something the current state lacks
(formation), not only for memory. The Blue control is there to show how big that effect is.

In [ ]:
SCAN = 5 + 0.375  # Red's target pick-up distance: minimum_scan_range + target radius
dmin_red = pd.DataFrame(np.fmin.reduce(gap, axis=1), index=steps)  # (steps, red units): nearest living Blue
close = dmin_red.le(SCAN) & dmin_red.notna()
was_close = (
    close.astype(float)
    .groupby(level=0)
    .transform(lambda x: x.rolling(8, min_periods=1).max().shift(1))
    .fillna(0)
    .astype(bool)
)
fire_red = pd.DataFrame(firing["red"], index=steps)
band = dmin_red.gt(SCAN) & dmin_red.le(REACH)

cases = {
    "within 5.375": close,
    "band, was within 5.375 in the last 1 s": band & was_close,
    "band, was not": band & ~was_close,
}
unit_test = pd.DataFrame(
    {
        name: {
            "unit-steps": int(m.to_numpy().sum()),
            "share firing": fire_red.to_numpy()[m.to_numpy()].mean(),
        }
        for name, m in cases.items()
    }
).T
unit_test.round(3)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor  # noqa: E402

LAG = 9
CURRENT = [
    "reach_blue_sm",
    "reach_red_sm",
    "alive_blue",
    "alive_red",
    "distance",
    "d_distance",
    *SHAPE,
    "hold",
    "withdraw",
    "focus",
]


def team_features(df):
    df = df.copy()
    if "d_distance" not in df:
        df = add_rates(df, cols=("distance",))
    g = df.groupby(level=0)
    for c in ("reach_blue_sm", "reach_red_sm", "distance", "firing_blue_sm", "firing_red_sm"):
        df[f"{c}_lag"] = g[c].shift(LAG)
    return df.dropna(subset=["firing_red_sm_lag"])


trf, vaf = team_features(tr), team_features(va)
sets = {
    "current state": CURRENT,
    "+ past R": CURRENT + ["reach_blue_sm_lag", "reach_red_sm_lag", "distance_lag"],
}
memory = []
for side in SIDES:
    y, yv = trf[f"firing_{side}_sm"].to_numpy(), vaf[f"firing_{side}_sm"].to_numpy()
    R, Rv = trf[f"reach_{side}_sm"].to_numpy(), vaf[f"reach_{side}_sm"].to_numpy()
    phi = (R @ y) / (R @ R)
    row = {"side": side, "share φ": phi, "share": r2(yv, phi * Rv)}
    for name, cols in {**sets, "+ past F": sets["+ past R"] + [f"firing_{side}_sm_lag"]}.items():
        m = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(trf[cols], y)
        row[name] = r2(yv, m.predict(vaf[cols]))
    memory.append(row)
memory = pd.DataFrame(memory).set_index("side")
memory["memory (past F − past R)"] = memory["+ past F"] - memory["+ past R"]
print("R² of smoothed F on the validation battles")
memory.round(3)

## Questions on test 2 (for you)

1. 2a: how big is the firing gap between "was close" and "was not" in the band, and how many
   unit-steps does it involve compared with all Red units in reach?
2. 2b: compare Red's memory gain with Blue's (the control). Is Red's clearly larger?
3. How far is the simple share from the flexible current-state model? Is the gap in φ's *shape*
   (it depends on the state), or in memory?
4. Taken together: state, share, or share with a few state-dependent terms?

## Test 3: does R need a state?

The same method as test 2, now on units in reach R itself (smoothed, train → validation):

- `closure`: `model_derived`'s A·Φ(d, depth, width). The algebraic closure the model uses today.
- `current state`: gradient-boosted trees on the current state *without* R: A, distance, d′,
  depth, width of both sides and the intent. This is about the best any closure of the current
  state could do.
- `+ past state`: the same, plus A, distance, depth and width 9 steps (1.1 s) earlier. This
  controls for "everything changes slowly".
- `+ past R`: the same, plus R 9 steps earlier.

Reading it:
- the gap `closure → current state` is what a better *closure shape* could gain;
- the gap `+ past state → + past R` is what *memory* (R as a state) could gain on top.

Caveat as before: past R can also stand in for finer formation detail that depth and width miss.

In [ ]:
PAST = ["alive_blue", "alive_red", "distance", *SHAPE]
NO_R = ["alive_blue", "alive_red", "distance", "d_distance", *SHAPE, "hold", "withdraw", "focus"]


def reach_features(df):
    df = df.copy()
    sb, sr = team_shares(df[STATE].to_numpy(), df[SHAPE].to_numpy(), current.P)
    df["closure_blue"], df["closure_red"] = df["alive_blue"] * sb, df["alive_red"] * sr
    g = df.groupby(level=0)
    for c in PAST + ["reach_blue_sm", "reach_red_sm"]:
        df[f"{c}_lag"] = g[c].shift(LAG)
    return df.dropna(subset=["reach_red_sm_lag"])


trr, var = reach_features(trf), reach_features(vaf)
reach_sets = {"current state": NO_R, "+ past state": NO_R + [f"{c}_lag" for c in PAST]}
reach_memory = []
for side in SIDES:
    y, yv = trr[f"reach_{side}_sm"].to_numpy(), var[f"reach_{side}_sm"].to_numpy()
    row = {"side": side, "closure": r2(yv, var[f"closure_{side}"].to_numpy())}
    for name, cols in {
        **reach_sets,
        "+ past R": reach_sets["+ past state"] + [f"reach_{side}_sm_lag"],
    }.items():
        m = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(trr[cols], y)
        pred = m.predict(var[cols])
        row[name] = r2(yv, pred)
        row[f"|err| units, {name}"] = REF_UNITS * np.abs(yv - pred).mean()
    row["|err| units, closure"] = REF_UNITS * np.abs(yv - var[f"closure_{side}"].to_numpy()).mean()
    reach_memory.append(row)
reach_memory = pd.DataFrame(reach_memory).set_index("side")
reach_memory["shape gain (current − closure)"] = reach_memory["current state"] - reach_memory["closure"]
reach_memory["memory gain (past R − past state)"] = reach_memory["+ past R"] - reach_memory["+ past state"]
print("R² of smoothed R on the validation battles (and mean |error| in units)")
reach_memory.T.round(3)

## Questions on test 3 (for you)

1. Which gap is larger for each side: the shape gain or the memory gain?
2. In units, how much of the closure's error could a better closure of the current state remove,
   and how much more would a state add?
3. What does that mean for the plan's "R as a state" step? Build the flows, improve the closure,
   or neither?

## Test 4: what limits the rollouts, the reach closure or the distance? (oracle check)

`model_derived` rolled out from every 8th step of the validation battles (as in the overview,
section 5), under the real intents, four ways:

- `model`: as it is;
- `true distance`: the real centroid distance fed in at every step (the best any distance model
  could do); the closure then works on the true distance, with formation held at its start value;
- `true distance + formation`: also the real depth and width (the closure on the true geometry,
  as in tests 1–3);
- `true alive`: the real number of units alive (R = A · share, so true R also carries who died);
- `true distance + formation + alive`: every input of the closure is real; what is left is the
  closure's *shape* error alone;
- `true R`: the real units in reach fed into the health equations (the best any closure *or*
  state for R could do). Caveat: it also carries the true deaths.

Metrics: health error (mean |error|, both sides), margin R² (change in Blue − Red from the start),
and units-in-reach error in units, at 1 / 2 / 4 / 8 s. 95% intervals for health 4 s and margin
4 s by resampling battles.

In [ ]:
from sindy_wm.models.intent_model import DT, make_batch  # noqa: E402

B = make_batch(val)
n_steps = B["U"].shape[1]
oracles = {
    "model": {},
    "true distance": {"distance": B["Y"][:, :, 4]},
    "true distance + formation": {"distance": B["Y"][:, :, 4], "shape": B["Yshape"]},
    "true alive": {"alive": B["Y"][:, :, 2:4]},
    "true distance + formation + alive": {
        "distance": B["Y"][:, :, 4],
        "shape": B["Yshape"],
        "alive": B["Y"][:, :, 2:4],
    },
    "true R": {"reach": B["Yreach"]},
}
roll = {}
for name, truth in oracles.items():
    Y, S = current.simulate(B["X0"], B["shape0"], B["U"], reach0=B["reach0"], truth=truth, return_shape=True)
    roll[name] = (Y, S)


def model_reach(Y, S):
    """Units in reach the model uses: closure on its own state, with the start offset decaying (tau = 1 s)."""
    N, T, _ = Y.shape
    s = np.stack(team_shares(Y.reshape(-1, 5), S.reshape(-1, 4), current.P), -1).reshape(N, T, 2)
    offset = B["reach0"] / np.maximum(B["X0"][:, 2:4], 1e-9) - s[:, 0]
    decay = np.exp(-np.arange(T) * DT / 1.0)[None, :, None]
    return Y[:, :, 2:4] * np.clip(s + offset[:, None] * decay, 0, 1)


def margin_r2(Y, n, idx=None):
    real = B["Y"][:, n, 0] - B["Y"][:, n, 1] - (B["Y"][:, 0, 0] - B["Y"][:, 0, 1])
    pred = Y[:, n, 0] - Y[:, n, 1] - (Y[:, 0, 0] - Y[:, 0, 1])
    ok = ~np.isnan(real)
    if idx is not None:
        ok = ok & idx
    return 1 - np.sum((pred[ok] - real[ok]) ** 2) / np.sum((real[ok] - real[ok].mean()) ** 2)


H = (1, 2, 4, 8)
oracle_table = {}
for name, (Y, S) in roll.items():
    R = B["Yreach"] if name == "true R" else model_reach(Y, S)
    row = {}
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(B["Y"][:, n, 0])
        row[f"health {h} s"] = np.abs(Y[ok, n, :2] - B["Y"][ok, n, :2]).mean()
    for h in H:
        row[f"margin R² {h} s"] = margin_r2(Y, round(h / DT))
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(B["Yreach"][:, n, 0])
        row[f"reach {h} s (units)"] = REF_UNITS * np.abs(R[ok, n] - B["Yreach"][ok, n]).mean()
    oracle_table[name] = row
oracle_table = pd.DataFrame(oracle_table).T

# bootstrap by battle: health 4 s and margin R² 4 s
rng = np.random.default_rng(0)
episodes, inverse = np.unique(B["ep"], return_inverse=True)
n4 = round(4 / DT)
ok4 = ~np.isnan(B["Y"][:, n4, 0])
picks = rng.integers(0, len(episodes), (500, len(episodes)))
for name, (Y, _) in roll.items():
    err = np.where(ok4, np.abs(Y[:, n4, :2] - B["Y"][:, n4, :2]).mean(1), 0)
    s, c = np.bincount(inverse, err, len(episodes)), np.bincount(inverse, ok4, len(episodes))
    boot = s[picks].sum(1) / c[picks].sum(1)
    oracle_table.loc[name, "health 4 s, 95%"] = (
        f"{np.quantile(boot, 0.025):.3f} – {np.quantile(boot, 0.975):.3f}"
    )
    real = B["Y"][:, n4, 0] - B["Y"][:, n4, 1] - (B["Y"][:, 0, 0] - B["Y"][:, 0, 1])
    pred = Y[:, n4, 0] - Y[:, n4, 1] - (Y[:, 0, 0] - Y[:, 0, 1])
    m = np.bincount(inverse, minlength=len(episodes))
    mb = []
    for p in picks[:200]:
        w = np.bincount(p, minlength=len(episodes))[inverse] * ok4
        w = np.where(ok4, w, 0)
        rr, pp = np.nan_to_num(real), np.nan_to_num(pred)
        mu = (w * rr).sum() / w.sum()
        mb.append(1 - (w * (pp - rr) ** 2).sum() / (w * (rr - mu) ** 2).sum())
    oracle_table.loc[name, "margin R² 4 s, 95%"] = (
        f"{np.quantile(mb, 0.025):.2f} – {np.quantile(mb, 0.975):.2f}"
    )
print(f"{len(B['X0'])} start points from {len(episodes)} battles")
oracle_table.T

## Questions on test 4 (for you)

1. Which oracle moves the health error at 4 s and the margin R² at 4 and 8 s the most?
2. With the true distance, how much reach error is left, and how does that compare with the
   closure's own error in tests 1–3 (about 0.5 units)?
3. Even with the true R, how far does the margin stay reliable? What does that leave as the limit?
4. So where should weeks 3–4 go: the closure, the distance equation, or consolidation?

## Test 5: a "wounded pool" alive rule

Team health H doesn't say how the damage is spread over the living units. On the common scale,
**W = A − H** is exactly the health missing from the units still alive (0 = everyone alive is at
full health). The mean wound per living unit is w = W / A, between 0 and 1. The more wounded the
survivors, the more of the next damage kills.

Candidate rule, SINDy (STLSQ, threshold 0.05) choosing among

    A' = H' · {1, w, focus, w·focus}

against today's `(ln A)' = p(u) · (ln H)'`, i.e. `A' = p(u) · (A/H) · H'`. Two physical bounds
are imposed in rollouts: deaths only (A never grows), and A ≥ H (a living unit holds at most one
unit of health).

Three checks: (a) per-step fit; (b) the alive rule driven by the *true* health (its own error,
as in the check before: about 0.7–0.8 units at 4 s today); (c) the full model rolled out with the
new rule.

In [ ]:
from sindy_wm.models.intent_model import CTRL, library  # noqa: E402

train_a = add_rates(
    add_controls(pd.read_parquet(PROCESSED / "train.parquet")),
    cols=("health_blue", "health_red", "alive_blue", "alive_red"),
)
val_a = add_rates(
    add_controls(pd.read_parquet(PROCESSED / "val.parquet")),
    cols=("health_blue", "health_red", "alive_blue", "alive_red"),
)


def both_alive(df):
    return df[
        (df["alive_blue"] > 0)
        & (df["alive_red"] > 0)
        & (df["health_blue"] > 0.01)
        & (df["health_red"] > 0.01)
    ]


train_a, val_a = both_alive(train_a), both_alive(val_a)
WOUND_TERMS = ["H'", "H'·w", "H'·focus", "H'·w·focus"]


def wound_features(H, A, dH, focus):
    w = np.clip((A - H) / np.maximum(A, 1e-9), 0, 1)
    return np.column_stack([dH, dH * w, dH * focus, dH * w * focus])


wound_coef, fit_rows = {}, []
for side in SIDES:

    def feats(df, side=side):
        return wound_features(
            df[f"health_{side}"].to_numpy(),
            df[f"alive_{side}"].to_numpy(),
            df[f"d_health_{side}"].to_numpy(),
            df["focus"].to_numpy(),
        )

    c = stlsq(feats(train_a), train_a[f"d_alive_{side}"].to_numpy(), 0.05)
    wound_coef[side] = c
    U_v = val_a[CTRL].to_numpy()
    old = (
        current._p_of(side, U_v)
        * val_a[f"alive_{side}"]
        / val_a[f"health_{side}"]
        * val_a[f"d_health_{side}"]
    )
    fit_rows.append(
        {
            "side": side,
            "wounded-pool rule": f"A_{side[0]}' = "
            + " ".join(f"{v:+.3f} {n}" for n, v in zip(WOUND_TERMS, c, strict=True) if v),
            "R² per step, today's rule": r2(val_a[f"d_alive_{side}"], old),
            "R² per step, wounded pool": r2(val_a[f"d_alive_{side}"], feats(val_a) @ c),
        }
    )
pd.DataFrame(fit_rows).set_index("side").round(3)

### (b) The alive rule on its own, driven by the true health

In [ ]:
CTRL_IDX = {"focus": 3}


def alive_step(rule, side, A, H0, H1, u):
    """One Euler step of A for one side, given health before (H0) and after (H1)."""
    if rule == "today":
        ratio = np.divide(H1, H0, out=np.zeros_like(H1), where=H0 > 1e-9)
        return A * ratio ** current._p_of(side, u)
    dH = H1 - H0  # = DT · H'
    A1 = A + wound_features(H0, A, dH, u[:, 3]) @ wound_coef[side]
    return np.clip(A1, np.minimum(H1, A), A)  # deaths only, and A ≥ H


Yt = B["Y"]
alive_check = {}
for rule in ("today", "wounded pool"):
    A = B["X0"][:, 2:4].copy()
    traj = [A.copy()]
    for n in range(B["U"].shape[1]):
        H0 = _h0 = np.nan_to_num(Yt[:, n, :2])
        H1 = np.where(np.isnan(Yt[:, n + 1, :2]), H0, np.nan_to_num(Yt[:, n + 1, :2]))
        A = np.column_stack(
            [alive_step(rule, s, A[:, j], H0[:, j], H1[:, j], B["U"][:, n]) for j, s in enumerate(SIDES)]
        )
        traj.append(A.copy())
    traj = np.stack(traj, 1)
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(Yt[:, n, 2])
        e = REF_UNITS * np.abs(traj[ok, n] - Yt[ok, n, 2:4]).mean(0)
        alive_check[(rule, f"{h} s")] = {"blue": e[0], "red": e[1]}
print("alive error (units) with the TRUE health driving the rule")
pd.DataFrame(alive_check).T.unstack(0).round(2)

### (c) The full model with the wounded-pool rule

`model_derived` with only the alive update swapped (same health, distance and closure), rolled
out from the same 8,946 start points as test 4.

In [ ]:
def simulate_alive_rule(rule, X0, S0, U, reach0):
    """`IntentModel.simulate` with a choice of alive rule.

    Shape held; the start reach offset decays with tau = 1 s.
    """
    X, S = X0.astype(float).copy(), S0.astype(float).copy()
    out = [X.copy()]
    Ch = np.stack([current.coef["health_blue"], current.coef["health_red"]])
    offset = reach0 / np.maximum(X[:, 2:4], 1e-9) - np.column_stack(team_shares(X, S, current.P))
    for n in range(U.shape[1]):
        u = U[:, n]
        s = np.column_stack(team_shares(X, S, current.P))
        reach = X[:, 2:4] * np.clip(s + offset * np.exp(-n * DT / 1.0), 0, 1)
        F_ = library(X, u, S, current.P, reach, current.ctrl)
        theta = np.column_stack([F_[k] for k in current.names])
        Hn = np.maximum(X[:, :2] + DT * theta @ Ch.T, 0)
        A = np.column_stack(
            [alive_step(rule, side, X[:, 2 + j], X[:, j], Hn[:, j], u) for j, side in enumerate(SIDES)]
        )
        X = np.column_stack([Hn, A, np.maximum(X[:, 4] + DT * theta @ current.coef["distance"], 0)])
        out.append(X.copy())
    return np.stack(out, 1)


def interval(values_per_point, ok, stat="mean", n_boot=500):
    """95% interval by resampling battles: mean of a per-start-point error, or margin R²."""
    lo_hi = []
    for p in picks[:n_boot]:
        wgt = np.bincount(p, minlength=len(episodes))[inverse] * ok
        if stat == "mean":
            lo_hi.append((wgt * np.nan_to_num(values_per_point)).sum() / wgt.sum())
        else:
            real, pred = (np.nan_to_num(v) for v in values_per_point)
            mu = (wgt * real).sum() / wgt.sum()
            lo_hi.append(1 - (wgt * (pred - real) ** 2).sum() / (wgt * (real - mu) ** 2).sum())
    return f"{np.quantile(lo_hi, 0.025):.3f} – {np.quantile(lo_hi, 0.975):.3f}"


full = {}
for rule in ("today", "wounded pool"):
    Y = (
        roll["model"][0]
        if rule == "today"
        else simulate_alive_rule(rule, B["X0"], B["shape0"], B["U"], B["reach0"])
    )
    if rule == "today":  # same simulator for both, so the comparison is like for like
        Y = simulate_alive_rule(rule, B["X0"], B["shape0"], B["U"], B["reach0"])
    row = {}
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(Yt[:, n, 0])
        row[f"health {h} s"] = np.abs(Y[ok, n, :2] - Yt[ok, n, :2]).mean()
    for h in H:
        row[f"margin R² {h} s"] = margin_r2(Y, round(h / DT))
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(Yt[:, n, 0])
        row[f"alive {h} s (units)"] = REF_UNITS * np.abs(Y[ok, n, 2:4] - Yt[ok, n, 2:4]).mean()
    err4 = np.abs(Y[:, n4, :2] - Yt[:, n4, :2]).mean(1)
    row["health 4 s, 95%"] = interval(err4, ok4)
    real4 = Yt[:, n4, 0] - Yt[:, n4, 1] - (Yt[:, 0, 0] - Yt[:, 0, 1])
    pred4 = Y[:, n4, 0] - Y[:, n4, 1] - (Y[:, 0, 0] - Y[:, 0, 1])
    row["margin R² 4 s, 95%"] = interval((real4, pred4), ok4, stat="r2", n_boot=200)
    full[rule] = row
pd.DataFrame(full)

## Questions on test 5 (for you)

1. Which terms did SINDy keep, and do the signs make physical sense? (More wounded survivors
   should mean more deaths per unit of health lost.)
2. In (b), how much of the alive rule's own error does the wounded pool remove?
3. In (c), does that carry through to the health error and the margin? If (b) improves but (c)
   doesn't, where does the gain get lost?

## Test 6: is there a floor on the alive error? (discreteness)

If deaths are mostly *timing noise* (which unit gets shot), no rule on team quantities can do
much better than today's. Upper bound: gradient-boosted trees that predict the number alive h
seconds ahead from everything a team-level model could ever know, *plus the true health path*:

- at the start: H, A, w = (A − H)/A for both sides, distance, depth, width;
- the true health of both sides at h/4, h/2 and h (the path, not just the end);
- the share of the horizon under focus and under withdraw;
- both alive rules' own predictions (so the trees can't do worse than them).

Trained on start points from the training battles, scored on the validation start points of
tests 4–5. If the trees also stay near 0.7–0.8 units at 4 s, that is the floor.

In [ ]:
B_train = make_batch(trf.pipe(lambda d: add_controls(pd.read_parquet(PROCESSED / "train.parquet"))))


def rule_paths(Bx):
    """Alive trajectories of today's rule and the wounded-pool rule, driven by the true health."""
    out = {}
    for rule in ("today", "wounded pool"):
        A = Bx["X0"][:, 2:4].copy()
        traj = [A.copy()]
        for n in range(Bx["U"].shape[1]):
            H0 = np.nan_to_num(Bx["Y"][:, n, :2])
            H1 = np.where(np.isnan(Bx["Y"][:, n + 1, :2]), H0, np.nan_to_num(Bx["Y"][:, n + 1, :2]))
            A = np.column_stack(
                [alive_step(rule, s, A[:, j], H0[:, j], H1[:, j], Bx["U"][:, n]) for j, s in enumerate(SIDES)]
            )
            traj.append(A.copy())
        out[rule] = np.stack(traj, 1)
    return out


def floor_features(Bx, paths, n, j):
    Yx = Bx["Y"]
    Hpath = np.where(np.isnan(Yx[:, :, :2]), np.nan, Yx[:, :, :2])
    Hpath = (
        pd.DataFrame(Hpath.reshape(len(Yx), -1)).T.ffill().T.to_numpy().reshape(Hpath.shape)
    )  # carry after the end
    X0, S0 = Bx["X0"], Bx["shape0"]
    w0 = np.clip((X0[:, 2:4] - X0[:, :2]) / np.maximum(X0[:, 2:4], 1e-9), 0, 1)
    cols = [X0[:, :5], w0, S0]
    for q in (n // 4, n // 2, n):
        cols.append(Hpath[:, q])
    cols.append(Bx["U"][:, :n, 3].mean(1, keepdims=True))
    cols.append(Bx["U"][:, :n, 2].mean(1, keepdims=True))
    cols.append(np.column_stack([paths["today"][:, n, j], paths["wounded pool"][:, n, j]]))
    return np.column_stack(cols)


paths_tr, paths_va = rule_paths(B_train), rule_paths(B)
floor = {}
for h in H:
    n = round(h / DT)
    ok_tr, ok_va = ~np.isnan(B_train["Y"][:, n, 2]), ~np.isnan(B["Y"][:, n, 2])
    for j, side in enumerate(SIDES):
        Xtr, Xva = floor_features(B_train, paths_tr, n, j), floor_features(B, paths_va, n, j)
        m = HistGradientBoostingRegressor(max_iter=300, random_state=0, loss="absolute_error").fit(
            Xtr[ok_tr], B_train["Y"][ok_tr, n, 2 + j]
        )
        truth_a = B["Y"][ok_va, n, 2 + j]
        floor[(f"{h} s", side)] = {
            "today's rule": REF_UNITS * np.abs(paths_va["today"][ok_va, n, j] - truth_a).mean(),
            "wounded pool": REF_UNITS * np.abs(paths_va["wounded pool"][ok_va, n, j] - truth_a).mean(),
            "trees (floor estimate)": REF_UNITS * np.abs(m.predict(Xva[ok_va]) - truth_a).mean(),
            "trees, rounded to whole units": REF_UNITS
            * np.abs(np.round(REF_UNITS * m.predict(Xva[ok_va])) / REF_UNITS - truth_a).mean(),
        }
floor = pd.DataFrame(floor).T
floor.index.names = ["horizon", "side"]
print(f"{ok_tr.sum()} training / {ok_va.sum()} validation start points at 8 s")
print("alive error in units, given the TRUE health path")
floor.round(2)

## Questions on test 6 (for you)

1. How close are the trees to today's rule? What does that say about how much any alive rule
   could still gain?
2. If the floor is close, how would you phrase the limit on the reliable horizon in the thesis?

## Test 7: refit the alive rule on multi-step error

Today's rule is biased: given the true health it predicts about 0.5 units too many alive at 4 s.
Hypothesis: attenuation. The rule is fitted on per-step *rates* `(ln A)'` against `(ln H)'`, both
noisy smoothed derivatives, and noise in the input pulls a least-squares coefficient towards zero
(too few deaths).

Fix: keep the **same terms** SINDy chose, and choose their coefficients so that the alive count
*simulated from the true health* matches the real one at 1, 2, 3 and 4 s (squared error, training
start points). The wounded-pool rule gets the same treatment, for comparison.

In [ ]:
from scipy.optimize import minimize  # noqa: E402

FIT_STEPS = [8, 16, 24, 32]


def alive_path(Bx, side, rule, params, steps=None):
    """A of one side simulated from the true health path of batch Bx, for given coefficients."""
    j = SIDES.index(side)
    steps = steps or Bx["U"].shape[1]
    A = Bx["X0"][:, 2 + j].copy()
    out = [A.copy()]
    for n in range(steps):
        H0 = np.nan_to_num(Bx["Y"][:, n, j])
        H1 = np.where(np.isnan(Bx["Y"][:, n + 1, j]), H0, np.nan_to_num(Bx["Y"][:, n + 1, j]))
        focus = Bx["U"][:, n, 3]
        if rule == "today":
            p = params[0] + (params[1] * focus if len(params) > 1 else 0)
            ratio = np.divide(H1, H0, out=np.zeros_like(H1), where=H0 > 1e-9)
            A = A * ratio**p
        else:
            coef = np.zeros(4)
            coef[: len(params)] = params
            A1 = A + wound_features(H0, A, H1 - H0, focus) @ coef
            A = np.clip(A1, np.minimum(H1, A), A)
        out.append(A.copy())
    return np.stack(out, 1)


def multistep_loss(params, Bx, side, rule):
    path = alive_path(Bx, side, rule, params, steps=max(FIT_STEPS))
    j = SIDES.index(side)
    loss = 0.0
    for n in FIT_STEPS:
        ok = ~np.isnan(Bx["Y"][:, n, 2 + j])
        loss += np.mean((path[ok, n] - Bx["Y"][ok, n, 2 + j]) ** 2)
    return loss


refit = {}
start = {
    ("blue", "today"): [current.p["blue"]["1"]],
    ("red", "today"): [current.p["red"]["1"], current.p["red"]["focus"]],
    ("blue", "wounded pool"): wound_coef["blue"][:3],
    ("red", "wounded pool"): wound_coef["red"][:3],
}
coef_rows = []
for (side, rule), p0 in start.items():
    res = minimize(
        multistep_loss,
        np.asarray(p0, float),
        args=(B_train, side, rule),
        method="Nelder-Mead",
        options={"xatol": 1e-4, "fatol": 1e-9, "maxiter": 2000},
    )
    refit[(side, rule)] = res.x
    names = ["p", "p·focus"] if rule == "today" else WOUND_TERMS[:3]
    coef_rows.append(
        {
            "side": side,
            "rule": rule,
            "fitted on rates": ", ".join(f"{n} {v:.3f}" for n, v in zip(names, p0, strict=False)),
            "fitted on multi-step error": ", ".join(
                f"{n} {v:.3f}" for n, v in zip(names, res.x, strict=False)
            ),
        }
    )
pd.DataFrame(coef_rows).set_index(["side", "rule"])

### The alive rule driven by the true health (validation), before and after the refit

In [ ]:
bias_rows = {}
for rule in ("today", "wounded pool"):
    for fitted, params_of in (
        ("rates", lambda s, r: start[(s, r)]),
        ("multi-step", lambda s, r: refit[(s, r)]),
    ):
        for side in SIDES:
            j = SIDES.index(side)
            path = alive_path(B, side, rule, params_of(side, rule))
            for h in H:
                n = round(h / DT)
                ok = ~np.isnan(B["Y"][:, n, 2 + j])
                e = REF_UNITS * (path[ok, n] - B["Y"][ok, n, 2 + j])
                bias_rows[(rule, fitted, side, f"{h} s")] = {
                    "mean |error|": np.abs(e).mean(),
                    "mean error (bias)": e.mean(),
                }
bias_table = pd.DataFrame(bias_rows).T
bias_table.index.names = ["rule", "fitted on", "side", "horizon"]
print("alive error in units given the TRUE health; floor estimate (test 6) at 4 s: 0.48 / 0.46")
bias_table.unstack("horizon").round(2)

### The full model with the refitted rule (validation rollouts, as in tests 4–5)

In [ ]:
def simulate_with_alive(alive_fn, X0, S0, U, reach0):
    """`simulate_alive_rule` with any alive update alive_fn(side, A, H0, H1, u)."""
    X, S = X0.astype(float).copy(), S0.astype(float).copy()
    out = [X.copy()]
    Ch = np.stack([current.coef["health_blue"], current.coef["health_red"]])
    offset = reach0 / np.maximum(X[:, 2:4], 1e-9) - np.column_stack(team_shares(X, S, current.P))
    for n in range(U.shape[1]):
        u = U[:, n]
        s = np.column_stack(team_shares(X, S, current.P))
        reach = X[:, 2:4] * np.clip(s + offset * np.exp(-n * DT / 1.0), 0, 1)
        F_ = library(X, u, S, current.P, reach, current.ctrl)
        theta = np.column_stack([F_[k] for k in current.names])
        Hn = np.maximum(X[:, :2] + DT * theta @ Ch.T, 0)
        A = np.column_stack(
            [alive_fn(side, X[:, 2 + j], X[:, j], Hn[:, j], u) for j, side in enumerate(SIDES)]
        )
        X = np.column_stack([Hn, A, np.maximum(X[:, 4] + DT * theta @ current.coef["distance"], 0)])
        out.append(X.copy())
    return np.stack(out, 1)


def today_refit(side, A, H0, H1, u):
    prm = refit[(side, "today")]
    p = prm[0] + (prm[1] * u[:, 3] if len(prm) > 1 else 0)
    ratio = np.divide(H1, H0, out=np.zeros_like(H1), where=H0 > 1e-9)
    return A * ratio**p


def wound_refit(side, A, H0, H1, u):
    coef = np.zeros(4)
    coef[:3] = refit[(side, "wounded pool")]
    return np.clip(A + wound_features(H0, A, H1 - H0, u[:, 3]) @ coef, np.minimum(H1, A), A)


variants = {
    "today (rates)": lambda side, A, H0, H1, u: alive_step("today", side, A, H0, H1, u),
    "today (multi-step)": today_refit,
    "wounded pool (multi-step)": wound_refit,
}
full7 = {}
for name, fn in variants.items():
    Y = simulate_with_alive(fn, B["X0"], B["shape0"], B["U"], B["reach0"])
    row = {}
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(Yt[:, n, 0])
        row[f"health {h} s"] = np.abs(Y[ok, n, :2] - Yt[ok, n, :2]).mean()
    for h in H:
        row[f"margin R² {h} s"] = margin_r2(Y, round(h / DT))
    for h in H:
        n = round(h / DT)
        ok = ~np.isnan(Yt[:, n, 0])
        row[f"alive {h} s (units)"] = REF_UNITS * np.abs(Y[ok, n, 2:4] - Yt[ok, n, 2:4]).mean()
    row["health 4 s, 95%"] = interval(np.abs(Y[:, n4, :2] - Yt[:, n4, :2]).mean(1), ok4)
    real4 = Yt[:, n4, 0] - Yt[:, n4, 1] - (Yt[:, 0, 0] - Yt[:, 0, 1])
    pred4 = Y[:, n4, 0] - Y[:, n4, 1] - (Y[:, 0, 0] - Y[:, 0, 1])
    row["margin R² 4 s, 95%"] = interval((real4, pred4), ok4, stat="r2", n_boot=200)
    full7[name] = row
pd.DataFrame(full7)

## Questions on test 7 (for you)

1. How much did the coefficients move, and in which direction? Does that fit attenuation?
2. Given the true health, how close does the refitted rule get to the floor (0.48 / 0.46 at 4 s)?
3. In the full model, does the refit carry through to health and margin? If not, why not?